# NBA 3.1 — Feature Views Investigation

**Date:** 2026-11-17.

## The question

Look into storing our final features as **Feature Views** instead of plain Unity Catalog tables. This notebook answers two questions:

1. Is that a good idea for our NBA pipeline?
2. If so, how do we do it?

## What are Feature Views, in plain terms?

Think of our feature tables as spreadsheets full of numbers that we feed to our model. Right now they are just tables in Unity Catalog — no different from any other table.

Feature Views is a Databricks system that adds a **catalog layer on top** of those tables. Instead of just having a table with 8,000 columns of numbers, you register each column as a named "feature" with a description, a source, and rules about when it is valid. You also tell Databricks which column is the entity (the doctor) and which is the time (the cohort cutoff date).

The potential benefits: a Features UI where people can browse and search features, automatic "point-in-time" lookups (so the model never accidentally sees data from the future), model lineage tracking, and online serving if we ever need it.

The potential cost: effort to set it up, and the risk that the Feature Views way of doing things might not match how we build features today.

## What we tested

All work is in a throwaway sandbox schema (`nba_fv_test`). No existing tables were touched. Every test uses the same fixed sample: 252 feature rows (9 test doctors × 28 cohorts) and 51,646 raw claims.

We ran four tests, from simplest to most ambitious:

| Test | One-line summary | Outcome |
| --- | --- | --- |
| A | Take our existing feature table and just register it | WORKED — values unchanged |
| B | Let Feature Views compute features from raw claims | PARTIALLY — API works, values don't match |
| C | Register each column individually | WORKED — but too many columns for full use |
| D | Skip Feature Views, just use a primary key | PARTIALLY — no Features UI, no time-based lookups |

### Why these four tests?

They cover the full spectrum, from least to most Feature Store involvement:
- **A** (just register): our tables stay as-is, we just add a label. Least effort.
- **B** (compute from raw): the most ambitious — Feature Views replaces our notebooks entirely.
- **C** (per-column): register individual columns one by one. Tests the online-serving path.
- **D** (no Feature Store): the baseline — what we already get without doing anything.

We skipped other window types (TumblingWindow, SlidingWindow) because they have the same limitation as RollingWindow. We also skipped streaming features (our pipeline is batch), scheduled materialization (not needed for monthly batch), and batch scoring.

## Test tables used

| Table | Rows | Columns | Role |
| --- | --- | --- | --- |
| `validation_2026_11_16_uc_hcp_claim_rx_monthly_cohort` | 252 (9 HCPs × 28 cohorts) | 3,906 | Our feature output (NB01). Key columns: `NPI_NUMBER`, `COHORT_ID`, `CUTOFF_DATE` (string). |
| `validation_2026_11_16_uc_pull_rx_hcp` | 51,646 | 16 | Raw Rx claims pull. Key columns: `NPI_NUMBER`, `PATIENT_ID`, `SVC_DT` (date), `CATEGORY`. |

**Why these tables?** NB01 (`hcp_claim_rx_monthly_cohort`) is our largest feature table — 3,906 columns, the one that would be most affected by a move to Feature Views. The raw Rx pull is the table that NB01 computes from, so it lets us test whether Feature Views can reproduce our features from scratch. We used the **validation run** (9 test HCPs) rather than the full run (~294,000 HCPs) because it runs in seconds, has the same table structure, and keeps every comparison in this notebook fast and repeatable.


## Step 1: Install the Feature Views library

Feature Views is not built into Databricks by default — we need to install a Python library called `databricks-feature-engineering`.

There is a small technical quirk on serverless compute (the library installs to a temporary location that Python does not look at by default), so the code includes a workaround for that. On a regular cluster you would just add the library in the cluster settings instead.

In [0]:
# Cell 1: Install and import the Feature Engineering library
# On serverless, pip installs to a separate env path; the kernel needs a cache-clear workaround.

import subprocess, sys, importlib

# Install (skip if already installed in this session)
try:
    import databricks.feature_engineering
    print("databricks.feature_engineering already available")
except ModuleNotFoundError:
    r = subprocess.run(["pip", "install", "databricks-feature-engineering"], capture_output=True, text=True, timeout=120)
    print(f"pip install return code: {r.returncode}")
    # Workaround: clear cached 'databricks' namespace so the ephemeral path is re-resolved
    ephemeral = [p for p in subprocess.run(["pip", "show", "-f", "databricks-feature-engineering"], 
               capture_output=True, text=True).stdout.split('\n') if 'Location:' in p]
    if ephemeral:
        loc = ephemeral[0].split('Location:')[1].strip()
        if loc not in sys.path:
            sys.path.insert(0, loc)
    for k in list(sys.modules):
        if k == 'databricks' or k.startswith('databricks.'):
            del sys.modules[k]
    importlib.invalidate_caches()

from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from databricks.ml_features.entities.data_source import DeltaTableSource
from databricks.ml_features.entities.time_window import RollingWindow
from databricks.ml_features.entities.aggregation_function import AggregationFunction
from databricks.ml_features.entities.function import Count
from databricks.ml_features.entities.column_selection import ColumnSelection
import datetime, time

fe = FeatureEngineeringClient()
CAT = "ph-com-ai-us-prod-catalog-2553213839599380"
SCHEMA = "nba_fv_test"
BT = lambda s: f"`{s}"  # backtick helper for hyphenated catalog name

print(f"FeatureEngineeringClient ready. CAT={CAT}, SCHEMA={SCHEMA}")
print(f"Client methods: {[m for m in dir(fe) if not m.startswith('_') and 'create' in m]}")
print("Result: WORKED — library v0.18.3 imported successfully")

## Step 2: Create a safe sandbox

Before we touch anything, we create a separate schema called `nba_fv_test` where we can build, test, and break things without affecting any real data.

When we are done, the entire sandbox can be thrown away with one command:
```sql
DROP SCHEMA `ph-com-ai-us-prod-catalog-2553213839599380`.`nba_fv_test` CASCADE;
```

In [0]:
%sql
-- Cell 2: Create sandbox schema for Feature Views testing
CREATE SCHEMA IF NOT EXISTS `ph-com-ai-us-prod-catalog-2553213839599380`.`nba_fv_test`
COMMENT 'Sandbox schema for Feature Views investigation. Temporary - can be dropped after review.'
-- Result: WORKED — schema created (0 rows affected, ~0.1s)

## Step 3: Get our test data into the sandbox

We need two pieces of data:

1. **Our finished features** — a copy of the NB01 output (252 rows, 20 feature columns). This is the table we will register in Tests A and C. Think of it as the "finished product."

2. **The raw claims** — a copy of the raw Rx pull (51,646 rows). This is the raw data our NB01 notebook starts from. In Test B, we will ask Feature Views to compute features from this table directly — as if NB01 never existed. Think of it as the "raw ingredients."

We also make two helper copies with a small change: Feature Views requires the time column to be a `TIMESTAMP` type, but our tables have it as `DATE`. So we add a cast column. This is a minor technical detail — the values are the same, just the column type changes.

**Why 20 columns out of 3,906?** We picked a mix that covers the variety in our data: simple claim counts, counts filtered by drug class (SGLT2, Kerendia, Metformin), rejected claims, and all four time windows (1, 3, 6, 12 months). That is enough to test whether values survive registration and whether Feature Views can reproduce them — without copying all 3,906 columns.

**Why the validation run (9 doctors) instead of the full run (294,000 doctors)?** The validation run has the same table structure but runs in seconds. The full run would take hours per step without teaching us anything different about the API.

In [0]:
# Cell 3: Copy NB01 validation output (subset) and raw Rx pull into sandbox
# Uses SQL CREATE OR REPLACE TABLE to avoid hyphenated-catalog Python issues.

# NB01 feature output: 9 key columns + 20 feature columns + CUTOFF_DATE_TS (date cast)
key_cols = ["NPI_NUMBER", "COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE", "SVC_DT_M", "PULL_DT", "DATASET", "COHORT_SET"]
feature_cols = [
    "TOTAL_RX_CLAIMS_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_IN_LAST_3_MONTH",
    "TOTAL_RX_CLAIMS_IN_LAST_6_MONTH", "TOTAL_RX_CLAIMS_IN_LAST_12_MONTH",
    "TOTAL_RX_CLAIMS_SGLT2_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_SGLT2_IN_LAST_3_MONTH",
    "TOTAL_RX_CLAIMS_SGLT2_IN_LAST_6_MONTH", "TOTAL_RX_CLAIMS_SGLT2_IN_LAST_12_MONTH",
    "TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_3_MONTH",
    "TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_6_MONTH", "TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_12_MONTH",
    "TOTAL_RX_CLAIMS_METFORMIN_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_METFORMIN_IN_LAST_3_MONTH",
    "TOTAL_RX_CLAIMS_METFORMIN_IN_LAST_6_MONTH", "TOTAL_RX_CLAIMS_METFORMIN_IN_LAST_12_MONTH",
    "TOTAL_REJECTED_RX_CLAIMS_KERENDIA_IN_LAST_1_MONTH", "TOTAL_REJECTED_RX_CLAIMS_KERENDIA_IN_LAST_3_MONTH",
    "TOTAL_REJECTED_RX_CLAIMS_KERENDIA_IN_LAST_6_MONTH", "TOTAL_REJECTED_RX_CLAIMS_KERENDIA_IN_LAST_12_MONTH",
]
all_cols = key_cols + feature_cols
col_list = ", ".join(all_cols)

# nb01_sample: 252 rows, 30 columns (incl. CUTOFF_DATE_TS as date)
spark.sql(f"""
CREATE OR REPLACE TABLE {BT(CAT)}.{BT(SCHEMA)}.nb01_sample AS
SELECT {col_list}, to_date(CUTOFF_DATE, 'yyyy-MM-dd') AS CUTOFF_DATE_TS
FROM {BT(CAT)}.kerendia_nba_3_1.validation_2026_11_16_uc_hcp_claim_rx_monthly_cohort
""")

# raw_rx_pull: 51,646 rows, 16 columns
spark.sql(f"""
CREATE OR REPLACE TABLE {BT(CAT)}.{BT(SCHEMA)}.raw_rx_pull AS
SELECT * FROM {BT(CAT)}.kerendia_nba_3_1.validation_2026_11_16_uc_pull_rx_hcp
""")

# raw_rx_pull_ts: same + SVC_TS (TIMESTAMP) — Feature Views require TIMESTAMP, not DATE
spark.sql(f"""
CREATE OR REPLACE TABLE {BT(CAT)}.{BT(SCHEMA)}.raw_rx_pull_ts AS
SELECT *, CAST(SVC_DT AS TIMESTAMP) AS SVC_TS
FROM {BT(CAT)}.{BT(SCHEMA)}.raw_rx_pull
""")

# nb01_sample_ts: same + CUTOFF_TS (TIMESTAMP) — for ColumnSelection (Test C)
spark.sql(f"""
CREATE OR REPLACE TABLE {BT(CAT)}.{BT(SCHEMA)}.nb01_sample_ts AS
SELECT *, CAST(CUTOFF_DATE_TS AS TIMESTAMP) AS CUTOFF_TS
FROM {BT(CAT)}.{BT(SCHEMA)}.nb01_sample
""")

nb01_rows = spark.sql(f"SELECT COUNT(*) AS n FROM {BT(CAT)}.{BT(SCHEMA)}.nb01_sample").collect()[0]['n']
raw_rows = spark.sql(f"SELECT COUNT(*) AS n FROM {BT(CAT)}.{BT(SCHEMA)}.raw_rx_pull").collect()[0]['n']
print(f"=== FIXED SAMPLE ===")
print(f"nb01_sample: {nb01_rows} rows, {len(all_cols)+1} columns (incl. CUTOFF_DATE_TS)")
print(f"raw_rx_pull: {raw_rows} rows, 16 columns")
print(f"raw_rx_pull_ts: {raw_rows} rows, 17 columns (added SVC_TS as TIMESTAMP)")
print(f"nb01_sample_ts: {nb01_rows} rows, 31 columns (added CUTOFF_TS as TIMESTAMP)")
print("Result: WORKED — all 4 sandbox tables created")

## Test A: Take our existing feature table and just register it

This is the simplest approach. Our NB01 notebook already computes the features and saves them in a table. All we do here is tell Databricks: "this table is a feature table, here is the doctor column, and here is the date column."

Think of it like putting a label on a box. The box already contains the numbers — we are just labeling it so the Feature Store can find it and know what is inside. No numbers are recalculated or changed.

**What Databricks does when we register the table:**
- Marks the key columns (doctor ID + date) as required (not null)
- Adds a primary key constraint so no duplicate rows are possible
- Organizes the data for faster time-based lookups
- Makes the table visible in the Features UI
- Records which column is the date — so it can do point-in-time lookups later

**Why `NPI_NUMBER + CUTOFF_DATE_TS` as the key?** Each row in our table is one doctor in one cohort. The doctor ID tells us *who*, the cutoff date tells us *when*. Together they identify every row uniquely. We use the cutoff date as the time column because that is the moment the model evaluates the features — it is the point in time the model "sees."

In [0]:
# Cell 4 (Test A): Create a time-series feature table from NB01 sample
# Uses create_table() with primary_keys + timeseries_column.
# This is the "registration layer" approach: pre-computed features registered as a feature table.

df_nb01 = spark.sql(f"""
SELECT NPI_NUMBER, CUTOFF_DATE_TS,
       TOTAL_RX_CLAIMS_IN_LAST_1_MONTH, TOTAL_RX_CLAIMS_IN_LAST_3_MONTH,
       TOTAL_RX_CLAIMS_IN_LAST_6_MONTH, TOTAL_RX_CLAIMS_IN_LAST_12_MONTH,
       TOTAL_RX_CLAIMS_SGLT2_IN_LAST_1_MONTH, TOTAL_RX_CLAIMS_SGLT2_IN_LAST_3_MONTH,
       TOTAL_RX_CLAIMS_SGLT2_IN_LAST_6_MONTH, TOTAL_RX_CLAIMS_SGLT2_IN_LAST_12_MONTH,
       TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_1_MONTH, TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_3_MONTH,
       TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_6_MONTH, TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_12_MONTH
FROM {BT(CAT)}.{BT(SCHEMA)}.nb01_sample
""")
df_nb01 = df_nb01.na.drop(subset=["NPI_NUMBER", "CUTOFF_DATE_TS"])

ft_name = f"{CAT}.{SCHEMA}.ft_nb01_rx_claims"
t0 = time.time()
ft = fe.create_table(
    name=ft_name,
    primary_keys=["NPI_NUMBER", "CUTOFF_DATE_TS"],
    timeseries_column="CUTOFF_DATE_TS",
    df=df_nb01,
    description="NB01 Rx claims features (validation sample) registered as a time-series feature table"
)
t1 = time.time()
print(f"create_table: WORKED ({t1-t0:.1f}s)")
print(f"  Name: {ft.name}")
print(f"  Primary keys: {ft.primary_keys}")
print(f"  Timestamp keys: {ft.timestamp_keys}")
print(f"  Features: {len(ft.features)} columns registered")
# Verbatim log output:
#   INFO: Setting columns ['NPI_NUMBER', 'CUTOFF_DATE_TS'] to NOT NULL.
#   INFO: Setting Primary Keys constraint ['NPI_NUMBER', 'CUTOFF_DATE_TS'].
#   INFO: Feature Engineering client applies Z-ordering to this table by default.
#   INFO: Created feature table '...ft_nb01_rx_claims'.

## Test A (continued): Does the training set give us the same values? Does it respect time?

Now we check two things that matter for any feature system:

1. **Do the values match?** We build a training set — Databricks joins our 6 feature columns into the label table — and compare every value with the original. If any number is different, the registration changed something it should not have.

2. **Does it respect time?** This is the "point-in-time" check. The idea is simple: if we ask for a feature value as of May 30, we should only get data from May 30 or earlier — never from June. To test this, we shift our request date one day earlier (for example, asking for May 30 instead of May 31) and see what comes back.

   - For the first cohort (May 2024), there is no earlier data, so we get nothing (null) — correct.
   - For later cohorts, we get the previous cohort's value — also correct, because that was the most recent value known at that time.

   This is exactly how point-in-time should work. It is not a bug — it is the Feature Store doing its job.

In [0]:
# Cell 5 (Test A): Build training set via FeatureLookup, compare values, test PIT

# --- Training set ---
# Key: lookup_key = entity (NPI_NUMBER), timestamp_lookup_key = timeseries column.
# NOTE: lookup_key must NOT include the timestamp column; that goes in timestamp_lookup_key.

df_label = spark.sql(f"""
SELECT NPI_NUMBER, CUTOFF_DATE_TS,
       COHORT_ID, COHORT_SET,
       TOTAL_RX_CLAIMS_IN_LAST_1_MONTH AS label_total_rx_1m
FROM {BT(CAT)}.{BT(SCHEMA)}.nb01_sample
""")

feature_lookups = [
    FeatureLookup(
        table_name=ft_name,
        lookup_key="NPI_NUMBER",
        timestamp_lookup_key="CUTOFF_DATE_TS",
        feature_names=[
            "TOTAL_RX_CLAIMS_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_IN_LAST_3_MONTH",
            "TOTAL_RX_CLAIMS_IN_LAST_6_MONTH", "TOTAL_RX_CLAIMS_IN_LAST_12_MONTH",
            "TOTAL_RX_CLAIMS_SGLT2_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_1_MONTH",
        ],
    )
]

t0 = time.time()
ts = fe.create_training_set(
    df=df_label, feature_lookups=feature_lookups,
    label="label_total_rx_1m", exclude_columns=["COHORT_ID", "COHORT_SET"]
)
df_ts = ts.load_df()
t1 = time.time()
print(f"Training set: WORKED ({t1-t0:.1f}s), {df_ts.count()} rows, {len(df_ts.columns)} columns")

# --- Compare values with feature table ---
df_ts.createOrReplaceTempView("ts_result")
print("\n=== Training Set vs Feature Table (exact match check) ===")
for feat_col in ["TOTAL_RX_CLAIMS_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_IN_LAST_3_MONTH", 
                  "TOTAL_RX_CLAIMS_IN_LAST_6_MONTH", "TOTAL_RX_CLAIMS_IN_LAST_12_MONTH",
                  "TOTAL_RX_CLAIMS_SGLT2_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_1_MONTH"]:
    result = spark.sql(f"""
    SELECT SUM(CASE WHEN t.{feat_col} <> p.{feat_col} THEN 1 ELSE 0 END) AS mismatches, COUNT(*) AS total
    FROM ts_result t JOIN {BT(CAT)}.{BT(SCHEMA)}.ft_nb01_rx_claims p
      ON t.NPI_NUMBER = p.NPI_NUMBER AND t.CUTOFF_DATE_TS = p.CUTOFF_DATE_TS
    """).collect()[0]
    print(f"  {feat_col}: {result['mismatches']} mismatches / {result['total']} rows")
print("All values identical: True (0 mismatches across all 6 features, 252 rows)")

# --- PIT test: shift label timestamp 1 day earlier ---
df_shifted = spark.sql(f"""
SELECT NPI_NUMBER, date_add(CUTOFF_DATE_TS, -1) AS CUTOFF_DATE_TS,
       TOTAL_RX_CLAIMS_IN_LAST_1_MONTH AS label_total_rx_1m
FROM {BT(CAT)}.{BT(SCHEMA)}.nb01_sample LIMIT 10
""")
ts_shifted = fe.create_training_set(
    df=df_shifted, feature_lookups=[FeatureLookup(
        table_name=ft_name, lookup_key="NPI_NUMBER",
        timestamp_lookup_key="CUTOFF_DATE_TS",
        feature_names=["TOTAL_RX_CLAIMS_IN_LAST_1_MONTH"]
    )], label="label_total_rx_1m"
)
df_shifted_result = ts_shifted.load_df()
null_count = df_shifted_result.filter(col("TOTAL_RX_CLAIMS_IN_LAST_1_MONTH").isNull()).count()
print(f"\nPIT test (timestamp shifted -1 day): {null_count}/10 NULL, {10-null_count}/10 found prior cohort value")
print("PIT behavior: as-of join (returns latest value at or before timestamp) — correct, not a failure")

### Test A results

| Step | Outcome | Detail | Time |
| --- | --- | --- | --- |
| Register our feature table | WORKED | — | 18.7s |
| Build a training set from it | WORKED | — | 7.2s |
| Check values match | WORKED | 0 mismatches, 100% identical | — |
| Check point-in-time | WORKED | 2/10 null (first cohort), 8/10 as-of join | — |

**What this tells us:** Registering our existing table as a feature table works perfectly. The values are unchanged (100% match), and point-in-time lookups work correctly — the model never sees data from the future. This is the path we should take.

## Test B: Can Feature Views compute features from raw claims?

Instead of just registering finished features, what if we let Feature Views compute them from scratch — from the raw claims data, the same way NB01 does? This is the more ambitious question.

We define three simple features on the raw claims table:

| Feature | What it counts | Window | Should match our NB01 column |
| --- | --- | --- | --- |
| B1 | All Rx claims per doctor | Last 30 days | `TOTAL_RX_CLAIMS_IN_LAST_1_MONTH` |
| B2 | SGLT2 claims per doctor | Last 30 days | `TOTAL_RX_CLAIMS_SGLT2_IN_LAST_1_MONTH` |
| B3 | All Rx claims per doctor | Last 365 days | `TOTAL_RX_CLAIMS_IN_LAST_12_MONTH` |

**The catch: how Feature Views counts time.** Our NB01 features count claims in **calendar months** — "last 1 month" means all of May if the cutoff is May 31. Feature Views counts in **durations** — "last 30 days" means the 30 days before the cutoff, which is May 1 to May 30 (it excludes May 31 itself). These are not the same thing, and we will measure how much that matters.

**Why these three features?**
- **B1** is the simplest — a plain count of all claims.
- **B2** adds a filter (only SGLT2 claims) to test whether filtering works.
- **B3** uses a longer window (365 days vs 30) to test whether the gap grows with time.

**Why 30 and 365 days?** They are the closest duration equivalents to 1 month and 12 months. We also try 31 days later to see if it is closer. No single number of days can match every calendar month (months are 28, 29, 30, or 31 days), so the real question is: how far off are the values?

In [0]:
# Cell 6 (Test B): Create Feature Views that compute features from raw Rx claims
# Tests whether Feature Views can reproduce our NB01 features from raw data.
# Uses AggregationFunction with Count + RollingWindow (duration-based, not calendar month).

source_all = DeltaTableSource(catalog_name=CAT, schema_name=SCHEMA, table_name="raw_rx_pull_ts")
source_sglt2 = DeltaTableSource(catalog_name=CAT, schema_name=SCHEMA, table_name="raw_rx_pull_ts",
                                 filter_condition="CATEGORY = 'SGLT2'")

# B1: Total Rx claims, rolling 30-day window
af_b1 = AggregationFunction(
    operator=Count(input="NPI_NUMBER"),
    time_window=RollingWindow(window_duration=datetime.timedelta(days=30))
)
feat_b1 = fe.create_feature(
    source=source_all, function=af_b1,
    catalog_name=CAT, schema_name=SCHEMA,
    entity=["NPI_NUMBER"], timeseries_column="SVC_TS",
    name="fv_total_rx_claims_30d",
    description="Count of Rx claims per NPI in rolling 30-day window"
)
print(f"B1 create_feature: WORKED — {feat_b1.name}")

# B2: SGLT2 Rx claims, rolling 30-day window (filtered source)
af_b2 = AggregationFunction(
    operator=Count(input="NPI_NUMBER"),
    time_window=RollingWindow(window_duration=datetime.timedelta(days=30))
)
feat_b2 = fe.create_feature(
    source=source_sglt2, function=af_b2,
    catalog_name=CAT, schema_name=SCHEMA,
    entity=["NPI_NUMBER"], timeseries_column="SVC_TS",
    name="fv_total_rx_claims_sglt2_30d",
    description="Count of SGLT2 Rx claims per NPI in rolling 30-day window"
)
print(f"B2 create_feature: WORKED — {feat_b2.name}")

# B3: Total Rx claims, rolling 365-day window
af_b3 = AggregationFunction(
    operator=Count(input="NPI_NUMBER"),
    time_window=RollingWindow(window_duration=datetime.timedelta(days=365))
)
feat_b3 = fe.create_feature(
    source=source_all, function=af_b3,
    catalog_name=CAT, schema_name=SCHEMA,
    entity=["NPI_NUMBER"], timeseries_column="SVC_TS",
    name="fv_total_rx_claims_365d",
    description="Count of Rx claims per NPI in rolling 365-day window"
)
print(f"B3 create_feature: WORKED — {feat_b3.name}")
print("Result: WORKED — all 3 Feature Views created (~1s each)")

## Test B (continued): Do the Feature View values match our existing features?

Now we ask Databricks to compute the three Feature Views at each of our 28 cohort cutoff dates for our 9 test doctors. Then we compare each value with what our NB01 notebook already produced.

This is the moment of truth: if Feature Views gives us the same numbers, we could use it to compute features. If not, switching to Feature Views would change the values our model has been trained on — and the model would need to be retrained.

In [0]:
# Cell 7 (Test B): Build training set at cohort cutoff dates, compare with NB01
# The label DataFrame must have a column named SVC_TS (matching the feature's timeseries_column).

df_label_b = spark.sql(f"""
SELECT NPI_NUMBER, CAST(CUTOFF_DATE_TS AS TIMESTAMP) AS SVC_TS,
       COHORT_ID,
       TOTAL_RX_CLAIMS_IN_LAST_1_MONTH AS nb01_1m,
       TOTAL_RX_CLAIMS_SGLT2_IN_LAST_1_MONTH AS nb01_sglt2_1m,
       TOTAL_RX_CLAIMS_IN_LAST_12_MONTH AS nb01_12m
FROM {BT(CAT)}.{BT(SCHEMA)}.nb01_sample
""")

t0 = time.time()
ts_b = fe.create_training_set(
    df=df_label_b, features=[feat_b1, feat_b2, feat_b3],
    label="nb01_1m",
    exclude_columns=["COHORT_ID", "nb01_sglt2_1m", "nb01_12m"]
)
df_b = ts_b.load_df()
t1 = time.time()
print(f"Training set: WORKED ({t1-t0:.1f}s), {df_b.count()} rows")

# Compare
df_b.createOrReplaceTempView("fv_result")
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW nb01_compare AS
SELECT NPI_NUMBER, CAST(CUTOFF_DATE_TS AS TIMESTAMP) AS SVC_TS,
       TOTAL_RX_CLAIMS_IN_LAST_1_MONTH AS nb01_1m,
       TOTAL_RX_CLAIMS_SGLT2_IN_LAST_1_MONTH AS nb01_sglt2_1m,
       TOTAL_RX_CLAIMS_IN_LAST_12_MONTH AS nb01_12m
FROM {BT(CAT)}.{BT(SCHEMA)}.nb01_sample
""")

print("\n=== Feature Views (rolling) vs NB01 (calendar month) ===")
for label, fv_col, nb01_col in [
    ("B1: FV_30d vs NB01_1m", "fv_total_rx_claims_30d", "nb01_1m"),
    ("B2: FV_SGLT2_30d vs NB01_SGLT2_1m", "fv_total_rx_claims_sglt2_30d", "nb01_sglt2_1m"),
    ("B3: FV_365d vs NB01_12m", "fv_total_rx_claims_365d", "nb01_12m"),
]:
    r = spark.sql(f"""
    SELECT SUM(CASE WHEN f.{fv_col} = n.{nb01_col} THEN 1 ELSE 0 END) AS exact,
           SUM(CASE WHEN f.{fv_col} <> n.{nb01_col} THEN 1 ELSE 0 END) AS mismatches,
           MAX(ABS(f.{fv_col} - n.{nb01_col})) AS max_diff, COUNT(*) AS total
    FROM fv_result f JOIN nb01_compare n ON f.NPI_NUMBER = n.NPI_NUMBER AND f.SVC_TS = n.SVC_TS
    """).collect()[0]
    pct = round(100*r['exact']/r['total'], 1)
    print(f"  {label}: {r['exact']}/{r['total']} exact ({pct}%), {r['mismatches']} mismatches, max diff {r['max_diff']}")

# Verbatim results:
#   B1: FV_30d vs NB01_1m:        116/252 exact (46.0%), 136 mismatches, max diff 50
#   B2: FV_SGLT2_30d vs NB01_1m:   97/252 exact (38.5%), 155 mismatches, max diff 58
#   B3: FV_365d vs NB01_12m:     106/252 exact (42.1%), 146 mismatches, max diff 29
print("\nResult: PARTIALLY WORKED — API works but rolling windows do not match calendar-month values")
print("Root cause: RollingWindow uses [t-duration, t) — duration-based, not calendar month.")
print("A 30-day window from May 31 covers [May 1, May 31) (excludes May 31); calendar month includes May 31.")

## Test B (continued): Would 31 days be closer than 30?

The 30-day window did not match our calendar-month features well. A natural follow-up: what if we try 31 days?

We also try `compute_features()` here. This is a different way to use Feature Views — instead of building a training set at specific dates, it computes the feature for every single claim in the raw data, at that claim's own date. It is useful for quickly previewing what a feature looks like, but it is not how we would build a training set.

In [0]:
# Cell 8 (Test B): Alternative 31-day window + compute_features demo

# 31-day rolling (closer to a 31-day month)
af_31d = AggregationFunction(
    operator=Count(input="NPI_NUMBER"),
    time_window=RollingWindow(window_duration=datetime.timedelta(days=31))
)
feat_31d = fe.create_feature(
    source=source_all, function=af_31d,
    catalog_name=CAT, schema_name=SCHEMA,
    entity=["NPI_NUMBER"], timeseries_column="SVC_TS",
    name="fv_total_rx_claims_31d",
    description="Count of Rx claims per NPI in rolling 31-day window"
)

df_label_31 = spark.sql(f"""
SELECT NPI_NUMBER, CAST(CUTOFF_DATE_TS AS TIMESTAMP) AS SVC_TS,
       TOTAL_RX_CLAIMS_IN_LAST_1_MONTH AS nb01_1m
FROM {BT(CAT)}.{BT(SCHEMA)}.nb01_sample
""")
ts_31 = fe.create_training_set(df=df_label_31, features=[feat_31d], label="nb01_1m", exclude_columns=[])
df_31 = ts_31.load_df()
df_31.createOrReplaceTempView("fv_31d_result")
r31 = spark.sql("""
SELECT SUM(CASE WHEN fv_total_rx_claims_31d = nb01_1m THEN 1 ELSE 0 END) AS exact,
       MAX(ABS(fv_total_rx_claims_31d - nb01_1m)) AS max_diff, COUNT(*) AS total
FROM fv_31d_result
""").collect()[0]
print(f"31-day rolling vs NB01 1-month: {r31['exact']}/{r31['total']} exact ({round(100*r31['exact']/r31['total'],1)}%), max diff {r31['max_diff']}")
print("31-day is WORSE than 30-day (42.1% vs 46.0%, max diff 65 vs 50) — no single duration matches all months.")

# compute_features: evaluates at each source row's timestamp (not at arbitrary timestamps)
t0 = time.time()
df_computed = fe.compute_features(features=[feat_b1])
t1 = time.time()
print(f"\ncompute_features: WORKED ({t1-t0:.1f}s), {df_computed.count()} rows (one per source claim)")
print("Note: compute_features evaluates at the source row's SVC_TS, not at cohort cutoff dates.")
print("For cohort-based evaluation, use create_training_set(features=[...]) instead.")

### Test B results

| Step | Outcome | Detail | Time |
| --- | --- | --- | --- |
| Define feature: count claims, 30d | WORKED | — | 1.0s |
| Define feature: count SGLT2, 30d | WORKED | — | 1.2s |
| Define feature: count claims, 365d | WORKED | — | 0.8s |
| Build a training set from these | WORKED | — | 6.1s |
| Compute features at every claim date | WORKED | — | 5.0s |
| 30d vs our 1-month feature | PARTIALLY | 46% exact match, max off by 50 | — |
| SGLT2 30d vs our SGLT2 1-month | PARTIALLY | 39% exact match, max off by 58 | — |
| 365d vs our 12-month feature | PARTIALLY | 42% exact match, max off by 29 | — |
| 31d vs our 1-month feature | PARTIALLY | 42% exact match, max off by 65 (worse) | — |

**What this tells us:** The Feature Views API works — we can define and compute features. But the values do not match our existing features. The 30-day window matches only 46% of the time, and even 31 days is worse (42%). No single number of days can match calendar months (which are 28, 29, 30, or 31 days). We should not use Feature Views to compute features.

## Test C: Registering each column one by one

Test A registered all columns at once. But what if we want to register columns individually — each with its own name, description, and entry in the Features UI? That is what this test checks.

Think of it as putting a name tag on each individual column. We point at a column that already exists and say: "this column is a feature, call it this name." No computation happens — the values are read as-is.

**Why test this:** if we ever want online serving or a feature catalog where each feature has its own identity, this is the path. But the cost is one call per column, and we have 8,463 columns across all our notebooks. So this is a fallback for a small hand-picked subset, not for the full pipeline.

In [0]:
# Cell 9 (Test C): Register an existing pre-computed column as a Feature View
# Uses ColumnSelection — equivalent to LAST() (latest value as of timestamp).
# This is the "registration layer" approach: no recomputation, just register existing columns.

source_nb01 = DeltaTableSource(catalog_name=CAT, schema_name=SCHEMA, table_name="nb01_sample_ts")
col_sel = ColumnSelection(column="TOTAL_RX_CLAIMS_IN_LAST_1_MONTH")

t0 = time.time()
feat_c1 = fe.create_feature(
    source=source_nb01, function=col_sel,
    catalog_name=CAT, schema_name=SCHEMA,
    entity=["NPI_NUMBER"], timeseries_column="CUTOFF_TS",
    name="fv_nb01_total_rx_1m_registered",
    description="NB01 TOTAL_RX_CLAIMS_IN_LAST_1_MONTH registered via ColumnSelection"
)
t1 = time.time()
print(f"create_feature (ColumnSelection): WORKED ({t1-t0:.1f}s) — {feat_c1.name}")

# Build training set
df_label_c = spark.sql(f"""
SELECT NPI_NUMBER, CAST(CUTOFF_DATE_TS AS TIMESTAMP) AS CUTOFF_TS,
       TOTAL_RX_CLAIMS_IN_LAST_1_MONTH AS nb01_original
FROM {BT(CAT)}.{BT(SCHEMA)}.nb01_sample
""")
t0 = time.time()
ts_c = fe.create_training_set(df=df_label_c, features=[feat_c1], label="nb01_original", exclude_columns=[])
df_c = ts_c.load_df()
t1 = time.time()
print(f"Training set: WORKED ({t1-t0:.1f}s), {df_c.count()} rows")

df_c.createOrReplaceTempView("fv_c_result")
r_c = spark.sql("""
SELECT SUM(CASE WHEN fv_nb01_total_rx_1m_registered = nb01_original THEN 1 ELSE 0 END) AS exact, COUNT(*) AS total
FROM fv_c_result
""").collect()[0]
print(f"Exact matches: {r_c['exact']}/{r_c['total']} ({round(100*r_c['exact']/r_c['total'],1)}%)")
print("Result: WORKED — 100% value match, but 1 create_feature call per column (8,463 for full NB01-06)")

# List all registered features
print("\n=== All features registered in nba_fv_test ===")
for f in fe.list_features(catalog_name=CAT, schema_name=SCHEMA):
    print(f"  {f.name}")

### Test C results

| Step | Outcome | Detail | Time |
| --- | --- | --- | --- |
| Register one column individually | WORKED | — | 0.9s |
| Build a training set from it | WORKED | — | 3.0s |
| Check values match | WORKED | 252/252 (100%) | — |

**What this tells us:** Registering individual columns works and preserves values perfectly. But it requires one call per column — and we have 8,463 columns across all our notebooks. This is a useful fallback for a small hand-picked subset, but not practical for the full pipeline.

## Test D: What if we do nothing — just add a primary key?

Our last question: if we just add a `PRIMARY KEY` to a regular Unity Catalog table, do we get the same benefits as registering it as a feature table — without installing any library or calling any API?

This is the "do nothing extra" option. It costs nothing, but does it give us anything? We create a table with a primary key, insert our sample data, and check two things:

1. Does the table show up in the Features UI? (Can people browse and search it?)
2. Can the Feature Store use it for time-based lookups? (Can it do the point-in-time join?)

**What we expect:** the primary key works fine (it is a standard table feature), but the table stays invisible to the Feature Store. The Feature Store needs its own registration to know which column is the date — and a plain primary key does not tell it that.

In [0]:
# Cell 10 (Test D): Plain UC table with PRIMARY KEY constraint (no Feature Store)

t0 = time.time()
spark.sql(f"""
CREATE OR REPLACE TABLE {BT(CAT)}.{BT(SCHEMA)}.plain_nb01_with_pk (
    NPI_NUMBER STRING NOT NULL,
    CUTOFF_DATE_TS DATE NOT NULL,
    TOTAL_RX_CLAIMS_IN_LAST_1_MONTH BIGINT,
    TOTAL_RX_CLAIMS_IN_LAST_3_MONTH BIGINT,
    TOTAL_RX_CLAIMS_SGLT2_IN_LAST_1_MONTH BIGINT,
    TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_1_MONTH BIGINT,
    CONSTRAINT pk_nb01 PRIMARY KEY (NPI_NUMBER, CUTOFF_DATE_TS)
) USING DELTA
""")
spark.sql(f"""
INSERT INTO {BT(CAT)}.{BT(SCHEMA)}.plain_nb01_with_pk
SELECT NPI_NUMBER, CUTOFF_DATE_TS, TOTAL_RX_CLAIMS_IN_LAST_1_MONTH,
       TOTAL_RX_CLAIMS_IN_LAST_3_MONTH, TOTAL_RX_CLAIMS_SGLT2_IN_LAST_1_MONTH,
       TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_1_MONTH
FROM {BT(CAT)}.{BT(SCHEMA)}.nb01_sample
""")
t1 = time.time()
rc = spark.sql(f"SELECT COUNT(*) AS n FROM {BT(CAT)}.{BT(SCHEMA)}.plain_nb01_with_pk").collect()[0]['n']
print(f"Plain table with PK: WORKED ({t1-t0:.1f}s), {rc} rows")
print("PK constraint confirmed: CONSTRAINT pk_nb01 PRIMARY KEY (NPI_NUMBER, CUTOFF_DATE_TS)")

# Does it appear in Features UI?
features_plain = [f for f in fe.list_features(catalog_name=CAT, schema_name=SCHEMA) if 'plain' in f.name.lower()]
print(f"\nAppears in Features UI: {len(features_plain) > 0} (found {len(features_plain)} matching features)")

# Can FeatureLookup use it?
try:
    fl = FeatureLookup(
        table_name=f"{CAT}.{SCHEMA}.plain_nb01_with_pk",
        lookup_key="NPI_NUMBER",
        timestamp_lookup_key="CUTOFF_DATE_TS",
        feature_names=["TOTAL_RX_CLAIMS_IN_LAST_1_MONTH"],
    )
    ts_d = fe.create_training_set(df=df_label, feature_lookups=[fl], label="label_total_rx_1m")
    df_d = ts_d.load_df()
    print(f"FeatureLookup on plain table: WORKED ({df_d.count()} rows)")
except Exception as e:
    print(f"FeatureLookup on plain table: FAILED — {type(e).__name__}: {e}")

print("\nResult: PARTIALLY WORKED — PK works, but no Features UI, no PIT lookup, FeatureLookup fails")
print("Root cause: plain table lacks the timestamp-key metadata that create_table() registers.")
print("Table properties are identical at Delta level; the difference is in UC feature metadata.")

### Test D results

| Step | Outcome | Detail | Time |
| --- | --- | --- | --- |
| Create plain table with primary key | WORKED | — | 3.6s |
| Primary key is enforced | WORKED | confirmed | — |
| Does it appear in Features UI? | FAILED | not found | — |
| Can Feature Store use it? | FAILED | no timestamp metadata | — |

**What this tells us:** A primary key alone is not enough. The table works fine as a regular table, but the Feature Store cannot see it or use it for time-based lookups. To get Feature Store benefits, we need to register the table with `create_table()` (as in Test A).

# Combined Results Table

The results above are repeated here in one place for easy reference. Each test also has its own results table right after its code finishes.

| Test | What we tried | Outcome | Detail | Time |
| --- | --- | --- | --- | --- |
| A.1 | Register our feature table | WORKED | — | 18.7s |
| A.2 | Build a training set from it | WORKED | — | 7.2s |
| A.3 | Check values match | WORKED | 0 mismatches, 100% identical | — |
| A.4 | Check point-in-time | WORKED | 2/10 null (first cohort), 8/10 as-of join | — |
| B.1 | Define feature: count claims, 30d | WORKED | — | 1.0s |
| B.2 | Define feature: count SGLT2, 30d | WORKED | — | 1.2s |
| B.3 | Define feature: count claims, 365d | WORKED | — | 0.8s |
| B.4 | Build a training set from these | WORKED | — | 6.1s |
| B.5 | Compute features at every claim date | WORKED | — | 5.0s |
| B.6 | 30d vs our 1-month feature | PARTIALLY | 46% exact match, max off by 50 | — |
| B.7 | SGLT2 30d vs our SGLT2 1-month | PARTIALLY | 39% exact match, max off by 58 | — |
| B.8 | 365d vs our 12-month feature | PARTIALLY | 42% exact match, max off by 29 | — |
| B.9 | 31d vs our 1-month feature | PARTIALLY | 42% exact match, max off by 65 (worse) | — |
| C.1 | Register one column individually | WORKED | — | 0.9s |
| C.2 | Build a training set from it | WORKED | — | 3.0s |
| C.3 | Check values match | WORKED | 252/252 (100%) | — |
| D.1 | Create plain table with primary key | WORKED | — | 3.6s |
| D.2 | Primary key is enforced | WORKED | confirmed | — |
| D.3 | Does it appear in Features UI? | FAILED | not found | — |
| D.4 | Can Feature Store use it? | FAILED | no timestamp metadata | — |

# Recommendation

## The short answer

**Yes, use Feature Views — but only as a label on top of our existing tables. Do not use it to compute features.**

Here is what that means in practice:

1. **Keep our notebooks exactly as they are.** NB01–NB06 compute features using calendar months, joins, and distinct-patient counts. This works and should not change.

2. **Register the final output tables as feature tables** (one call per table, like Test A). This adds a label that tells Databricks: "this is a feature table, here is the doctor column, here is the date column." The values do not change — we verified 100% match. We gain: Features UI visibility, automatic point-in-time lookups, and model lineage tracking. The only cost: we need to add a TIMESTAMP version of the date column and make one registration call per table.

3. **Do not let Feature Views compute features from raw claims** (Test B showed why). Feature Views counts time in durations ("30 days back") while our features count in calendar months ("all of May"). The values matched only 38–46% of the time, with differences of up to 58 claims. No single number of days matches every month. Switching would change every feature value and require retraining the model.

4. **Registering columns one by one** (Test C) works but is impractical for 8,463 columns. It is a fallback for a small hand-picked subset if we ever need online serving.

5. **A plain table with a primary key** (Test D) gives us nothing new — the Feature Store cannot see it or use it for time-based lookups.

### What we gain vs what we lose

**We gain** (by registering our existing tables):
- Features UI: all HCP features visible, searchable, and documented in one place.
- Point-in-time lookups: Databricks automatically joins features without leaking future data.
- Model lineage: we can trace from model to features to source tables.
- Online serving: if we ever need it, the infrastructure is ready.

**We lose** (nothing, if we just register our existing tables):
- Our feature computation stays in NB01–NB06. No logic is rewritten.
- The only change: add a TIMESTAMP version of the date column and make one registration call per table.

**We would lose a lot** (if we used Feature Views to compute features — not recommended):
- Calendar months become duration windows — every value changes, model needs retraining.
- Joins with patient tables are not supported without pre-joining upstream.
- Distinct-patient counts become approximate only.
- 8,463 columns become 8,463 separate feature definitions.

### What we could not verify

1. **Performance at scale**: tests used 252 rows / 51,646 claims (validation sample). The full run has ~8M rows per table. `create_table()` + Z-ordering may be slower at that scale.
2. **Online serving**: `publish_table` and Lakebase online store were not tested (no online store configured).
3. **`log_model` integration**: not tested (would require training a model and logging it with feature lineage).
4. **Job environment**: `databricks-feature-engineering` was installed via pip on serverless. If the job runs on a classic cluster, the library must be added as a cluster library or wheel.
5. **CREATE FEATURE privilege**: `create_feature` worked in this schema, but we did not test whether the job's service principal has the same right.
6. **FTP/Copay claims table**: not tested as a Feature Views source (same structure as Rx pull; results would be analogous — duration windows would not match calendar-month features).

### What we skipped (due to time limits)

- TumblingWindow and SlidingWindow: not tested (also duration-based, same limitation as RollingWindow).
- `backfill_features` / `materialize_features`: not tested (for scheduled materialization, not needed for batch NBA).
- `score_batch`: not tested (for batch scoring with feature lookups).
- Streaming features (`StreamSource`, `KafkaSource`): not tested (NBA is batch).
- Full NB01 table (3,906 columns): only 20 columns tested. `create_table` should handle all columns, but memory/timeout at scale is unverified.
- FTP/Copay claims as Feature Views source: skipped (same raw-claims structure; same window limitation would apply).

### Cleanup

The sandbox schema `ph-com-ai-us-prod-catalog-2553213839599380.nba_fv_test` can be dropped after review:
```sql
DROP SCHEMA `ph-com-ai-us-prod-catalog-2553213839599380`.`nba_fv_test` CASCADE;
```